# 基于 LSTM 的股票价格走势预测实战

## 1. 内容概述

股票市场是一个复杂的、非线性的动态系统，其价格波动受到宏观经济、公司基本面、市场情绪等多种因素的影响。准确预测股票价格是金融领域最具挑战性的任务之一。传统的时间序列分析方法（如 ARIMA）依赖线性假设，难以捕捉股价数据中复杂的非线性关系和长期依赖特征。

循环神经网络（Recurrent Neural Networks, RNN）是专门处理序列数据的神经网络类型。与传统前馈网络不同，RNN 通过隐藏状态将历史信息传递到当前时间步，从而捕捉序列中的时序依赖。然而，传统 RNN 在处理长序列时会遇到**梯度消失**问题——梯度随时间步指数级衰减，导致模型无法学习远距离的依赖关系。

**长短期记忆网络（LSTM）** 通过引入门控机制解决了这一问题。LSTM 包含三个门：

- **遗忘门（Forget Gate）**：控制从细胞状态中丢弃哪些历史信息
- **输入门（Input Gate）**：决定哪些新信息被写入细胞状态
- **输出门（Output Gate）**：决定当前细胞状态输出哪些信息

这种选择性记忆机制使 LSTM 能够在长序列中有效保留关键信息、遗忘冗余信息，特别适合股价等具有长期依赖特征的时间序列预测任务。

本案例演示如何在昇腾 NPU 上，使用 **PyTorch** 深度学习框架（通过 `torch_npu` 适配层使能 NPU 加速）构建 **两层 LSTM** 模型，以"贵州茅台"（股票代码：600519）的历史日交易数据为样本，预测其未来收盘价走势，内容涵盖：

- CANN 环境加载与 NPU 可用性验证
- 使用 akshare 获取股票历史交易数据
- 数据归一化与时间窗口数据集构建
- LSTM 模型设计与代码实现
- 损失函数、优化器配置与模型训练
- 测试集预测与 RMSE 评估

## 2. 整体流程概览

本案例将完整走通在昇腾 NPU 上基于 LSTM 进行股价预测的全流程：

```
环境准备 → 数据获取 → 数据预处理（归一化 + 时间窗口） → 模型构建（LSTM） → 训练 → 测试集预测
```

各环节说明：

| 环节 | 说明 |
|------|------|
| 环境准备 | 加载 CANN 环境变量，安装 Python 依赖，验证 NPU 可用性 |
| 数据获取 | 通过 akshare 获取贵州茅台历史日交易数据，网络不通时从 AtomGit 下载离线数据 |
| 数据预处理 | 对收盘价归一化至 [0,1] 区间，用滑动窗口构建监督学习数据集，按 8:2 划分训练 / 测试集 |
| 模型构建 | 设计两层 LSTM + 全连接层的 StockLSTM 模型 |
| 训练 | 配置 MSELoss 与 Adam 优化器，手写训练循环在 NPU 上训练 10 轮 |
| 测试集预测 | 对测试集进行预测，反归一化后绘制对比图并计算 RMSE |

下面逐步实现每个环节。

## 3. 环境准备与依赖安装

本案例依赖以下核心组件：

| 库名 | 用途 |
|------|------|
| `torch` + `torch_npu` | PyTorch 深度学习框架 + 昇腾 NPU 适配层 |
| `akshare` | 获取 A 股历史交易数据 |
| `pandas` | 数据处理与日期解析 |
| `matplotlib` | 数据可视化与预测结果绘图 |
| `numpy` | 数值计算与指标计算 |
| `scikit-learn` | 数据归一化（MinMaxScaler）与 RMSE 评估 |

### 3.1 加载 CANN 环境

> ⚠️ **必须在 `import torch` 之前执行此单元格！**
> Jupyter 内核不继承 shell 环境变量，缺这步会导致 `torch_npu` 加载失败。

In [ ]:
import os
import subprocess
import warnings

warnings.filterwarnings('ignore')

# CANN 安装路径候选列表，按优先级尝试
_cann_paths = [
    os.path.join(os.environ.get("ASCEND_TOOLKIT_HOME", ""), "set_env.sh"),
    "/home/developer/Ascend/cann-9.2.0/set_env.sh",
    "/usr/local/Ascend/ascend-toolkit/set_env.sh",
]
_cann = next((p for p in _cann_paths if p and os.path.exists(p)), None)
if _cann is None:
    raise FileNotFoundError("找不到 CANN set_env.sh，请确认 CANN 安装路径")

# 通过 subprocess 执行 source 命令，将 CANN 环境变量注入当前内核
print(f"正在加载 CANN 环境: {_cann}")
result = subprocess.run(["bash", "-c", f"source {_cann} && env"], capture_output=True, text=True, timeout=10)
for line in result.stdout.splitlines():
    if "=" in line:
        k, v = line.split("=", 1)
        os.environ[k] = v

print(f"CANN 环境已加载 (ASCEND_TOOLKIT_HOME={os.environ.get('ASCEND_TOOLKIT_HOME','?')})")

### 3.2 安装 Python 依赖

若环境中已预装所需库，可跳过此步骤。

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "torch>=2.5", "torch_npu", "akshare>=1.12", "pandas>=2.0",
     "matplotlib>=3.7", "numpy>=1.24", "scikit-learn>=1.3"],
    check=True,
)

### 3.3 验证 NPU 环境

运行以下代码，确认昇腾 NPU 可用。如果输出 NPU 设备信息，说明环境配置正确。

> 💡 **小贴士**：`torch_npu` 是昇腾 NPU 的 PyTorch 适配层。导入它之后，PyTorch 就能自动识别并使用 NPU 进行计算，使用方式与 `torch.cuda` 非常相似。例如：
> - `torch.npu.set_device(0)` 等价于 `torch.cuda.set_device(0)`
> - `model.to("npu")` 等价于 `model.to("cuda")`
>
> 代码开头的 `ctypes.CDLL` 预加载步骤用于提前加载 torch / sklearn 底层的 `.so` 共享库，解决 NPU 环境下可能出现的符号冲突问题。

In [ ]:
# 预加载 torch / sklearn 的底层 .so 库，解决 NPU 环境下的符号冲突问题
import ctypes, glob, os, importlib.util
for pkg, sub in [("torch", "lib"), ("sklearn", "../scikit_learn.libs")]:
    spec = importlib.util.find_spec(pkg)
    if not spec or not spec.origin:
        continue
    d = os.path.join(os.path.dirname(spec.origin), sub)
    for so in glob.glob(os.path.join(d, "*.so*")):
        try: ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
        except OSError: pass

import torch
import torch_npu  # 注册 Ascend NPU 后端，使能 NPU 加速
device = torch.device('npu' if torch.npu.is_available() else 'cpu')

print(f"PyTorch 版本: {torch.__version__}")
print(f"当前设备: {device}")
if str(device) == "npu":
    print(f"NPU 设备数量: {torch.npu.device_count()}")
    print(f"当前 NPU 设备: {torch.npu.current_device()}")
    print(f"NPU 设备名称: {torch.npu.get_device_name(0)}")

## 4. 模型介绍

### 4.1 RNN 与 LSTM 简介

循环神经网络（Recurrent Neural Networks, RNN）是一类专门处理序列数据的神经网络。与传统的前馈神经网络不同，RNN 在处理序列时维护一个隐藏状态（hidden state），将之前时间步的信息传递到当前时间步，从而捕捉序列中的时序依赖关系。

然而，传统 RNN 在处理长序列时会遇到**梯度消失**问题——在反向传播时，梯度需要沿时间方向反复乘以权重矩阵，随着时间步增加，梯度会指数级衰减，导致模型无法学习长期依赖。

**长短期记忆网络（LSTM）** 由 Hochreiter 和 Schmidhuber 于 1997 年提出，通过引入门控机制解决了梯度消失问题。LSTM 的核心是细胞状态（cell state），信息通过三个门控制流动：

- **遗忘门（Forget Gate）**：读取当前输入与上一步隐藏状态，输出 0~1 之间的值，决定从细胞状态中保留多少历史信息。1 表示全部保留，0 表示全部遗忘
- **输入门（Input Gate）**：决定哪些新信息被写入细胞状态，与候选值向量配合更新细胞状态
- **输出门（Output Gate）**：基于更新后的细胞状态，决定当前时间步的隐藏状态输出什么信息

通过这三个门，LSTM 能够选择性地保留长期信息或遗忘无关信息，有效捕捉时间序列中的长期依赖关系，因此在语音识别、机器翻译、金融时间序列预测等任务中表现优异。

### 4.2 模型结构设计

本案例构建一个包含 **两层 LSTM** 和一个 **全连接输出层** 的模型，结构如下：

| 组件 | 说明 |
|------|------|
| `nn.LSTM` | PyTorch 的 LSTM 层，`batch_first=True` 表示输入第一个维度是 batch_size |
| `nn.Linear` | 全连接层，将 LSTM 最后一个时间步的隐藏状态映射到最终的预测值 |

模型的数据流如下：

```
输入 (batch_size, seq_length, 1) → 两层 LSTM → 取最后时间步输出 (batch_size, hidden_size) → 全连接层 → 预测值 (batch_size, 1)
```

**设计要点：**

- 输入维度 `input_size=1`：每个时间步只有一个特征（收盘价）
- 隐藏维度 `hidden_size=64`：LSTM 隐藏层大小，决定模型的记忆容量
- 层数 `num_layers=2`：两层堆叠 LSTM，第一层提取短期模式，第二层提取更高层次的时序特征
- 取最后时间步输出：LSTM 处理完整个 60 天序列后，用最后一个时间步的隐藏状态进行预测

### 4.3 模型代码实现

In [ ]:
import torch.nn as nn


class StockLSTM(nn.Module):
    """基于 LSTM 的股价预测模型

    结构：两层 LSTM → 取最后时间步输出 → 全连接层 → 预测值
    """

    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(StockLSTM, self).__init__()
        # LSTM 层：处理输入序列，返回所有时间步的输出和最终隐藏状态
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,  # 输入维度顺序: (batch, seq, feature)
        )
        # 全连接层：将 LSTM 输出映射到预测值
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # LSTM 返回 (output, (h_n, c_n))，output 形状为 (batch, seq, hidden)
        output, _ = self.lstm(x)
        # 只取最后一个时间步的输出，因为我们要预测的是"看完 60 天后下一天的收盘价"
        last_output = output[:, -1, :]  # (batch, seq, hidden) → (batch, hidden)
        prediction = self.fc(last_output)
        return prediction


# 实例化模型并迁移至设备
net = StockLSTM(input_size=1, hidden_size=64, num_layers=2, output_size=1).to(device)
print(net)

> 💡 **关键代码解读**：
> - `nn.Module`：PyTorch 所有模型的基类，模型参数通过 `.to(device)` 迁移到 NPU
> - `input_size=1`：每个时间步的输入特征维度为 1（仅收盘价）
> - `hidden_size=64`：LSTM 隐藏层维度，决定模型记忆容量
> - `num_layers=2`：两层堆叠 LSTM，提取多层次的时序特征
> - `output[:, -1, :]`：只取最后一个时间步的输出，因为预测目标是序列末尾之后的下一天价格

## 5. 数据集介绍与下载

### 5.1 数据集简介

本案例使用 **akshare** 库获取 A 股历史交易数据。选取 **贵州茅台**（股票代码：600519）的日交易数据作为实验样本，时间跨度从 2010 年 1 月 1 日至 2025 年 6 月 25 日，共约 3750 条记录。

数据包含日期、开盘价、收盘价、最高价、最低价、成交量等字段。本案例仅使用 **收盘价** 作为预测目标，将其处理为时间序列监督学习问题：

- **输入 X**：过去 60 天的收盘价序列
- **输出 y**：第 61 天的收盘价

这种滑动窗口方式将时间序列转化为标准的监督学习格式，模型学习的是"历史价格模式 → 未来价格"的映射关系。

### 5.2 获取股票数据

数据获取采用**双通道机制**，确保在不同网络环境下均可成功获取数据：

- **方式一（优先）**：通过 akshare 从东方财富 API 获取最新实时数据，适用于可访问外网的环境
- **方式二（兜底）**：akshare 失败时，自动从 AtomGit 仓库克隆离线 CSV 数据，适用于 CANNLab 等网络受限环境

> 💡 **说明**：CANNLab 等在线环境可能无法访问东方财富的服务器（`push2his.eastmoney.com`），此时 akshare 会失败，代码会自动切换到 AtomGit 下载离线数据，不影响后续训练流程。离线数据与实时数据格式完全一致。

In [ ]:
import akshare as ak
import time
import os
import subprocess

# 股票代码：600519 = 贵州茅台
stock_code = "600519"
# 数据时间范围：2010-01-01 ~ 2025-06-25
start_date = "20100101"
end_date = "20250625"

stock_df = None

# ---- 方式一：通过 akshare 从东方财富获取实时数据（需要可访问外网）----
# 设置 3 次重试，每次间隔 3 秒
max_retries = 3
for attempt in range(max_retries):
    try:
        print(f"正在通过 akshare 获取数据（第 {attempt + 1} 次尝试）...")
        stock_df = ak.stock_zh_a_hist(
            symbol=stock_code,       # 股票代码
            period="daily",          # 日K线数据
            start_date=start_date,   # 起始日期
            end_date=end_date,       # 结束日期
            adjust="qfq",            # 前复权（消除分红送股对价格的影响）
        )
        print(f"akshare 获取成功，数据条数: {len(stock_df)}")
        break
    except Exception as e:
        print(f"第 {attempt + 1} 次获取失败: {e}")
        if attempt < max_retries - 1:
            time.sleep(3)

# ---- 方式二：akshare 失败时，从 AtomGit 仓库克隆离线 CSV ----
if stock_df is None:
    print("\nakshare 获取失败，正在从 AtomGit 下载离线数据...")
    import pandas as pd
    repo_url = "https://atomgit.com/ink_polymer/maotai_daily_2010_2025.git"
    clone_dir = "/tmp/maotai_data"
    if not os.path.exists(clone_dir):
        subprocess.run(
            ["git", "clone", "--depth", "1", repo_url, clone_dir],
            check=True, capture_output=True, text=True, timeout=120,
        )
    csv_path = os.path.join(clone_dir, "maotai_daily_2010_2025.csv")
    stock_df = pd.read_csv(csv_path)
    print(f"离线数据下载成功，数据条数: {len(stock_df)}")

print(f"\n数据条数: {len(stock_df)}")
print(stock_df.head())

### 5.3 数据可视化

先看看收盘价的历史走势，直观感受数据特征。贵州茅台从 2010 年到 2025 年的股价经历了多次大幅波动，这种非线性、带趋势的时间序列正是 LSTM 擅长处理的场景。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# 将日期列转为 datetime 类型便于绘图
stock_df['日期'] = pd.to_datetime(stock_df['日期'])

plt.figure(figsize=(14, 5))
plt.plot(stock_df['日期'], stock_df['收盘'])
plt.title(f'{stock_code} Closing Price History')
plt.xlabel('Date')
plt.ylabel('Price')
plt.grid(True)
plt.show()

### 5.4 数据预处理

数据预处理分为三步：

1. **归一化**：使用 `sklearn.preprocessing.MinMaxScaler` 将收盘价缩放到 `[0, 1]` 区间，消除量纲差异，加速模型收敛。归一化公式为 `x_scaled = (x - min) / (max - min)`，`MinMaxScaler` 会自动记录 min/max 以便后续反归一化
2. **构建时间窗口**：通过滑动窗口将序列转化为 `(X, y)` 对，X 为过去 60 天的收盘价，y 为第 61 天的收盘价
3. **划分训练 / 测试集**：按 80% / 20% 的比例划分，前 80% 用于训练，后 20% 用于测试

**关键参数说明：**

| 参数 | 含义 |
|------|------|
| `seq_length=60` | 使用过去 60 天的数据预测下一天 |
| `train_size=0.8` | 80% 数据用于训练，20% 用于测试 |
| `feature_range=(0, 1)` | 归一化区间 |

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Step 1: 归一化 — 将收盘价缩放到 [0, 1] 区间
scaler = MinMaxScaler(feature_range=(0, 1))
close_prices = stock_df['收盘'].values.astype(np.float32)
scaled_prices = scaler.fit_transform(close_prices.reshape(-1, 1)).flatten()


# Step 2: 构建时间窗口数据集
# 滑动窗口：用第 i ~ i+59 天的价格作为输入，预测第 i+60 天的价格
def create_dataset(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:(i + seq_length)])  # 过去 seq_length 天
        y.append(data[i + seq_length])       # 第 seq_length+1 天
    return np.array(X), np.array(y)


seq_length = 60  # 使用过去 60 天的收盘价预测下一天
X, y = create_dataset(scaled_prices, seq_length)

# Step 3: 划分训练集和测试集（按时间顺序，前 80% 训练，后 20% 测试）
train_size = int(len(X) * 0.8)
X_train, X_test = X[0:train_size], X[train_size:]
y_train, y_test = y[0:train_size], y[train_size:]

print(f"训练集大小: {len(X_train)}")
print(f"测试集大小: {len(X_test)}")

### 5.5 转换为 PyTorch Tensor

RNN 输入需要 3D 张量 `(batch_size, seq_length, input_size)`。将 numpy 数组转换为 PyTorch Tensor，后续通过 `TensorDataset` 和 `DataLoader` 自动完成批量加载与设备迁移。

In [ ]:
# 转换为 PyTorch Tensor
# RNN 输入需要 3D 张量 (batch_size, seq_length, input_size)
X_train = torch.tensor(X_train.reshape(X_train.shape[0], X_train.shape[1], 1), dtype=torch.float32)
X_test = torch.tensor(X_test.reshape(X_test.shape[0], X_test.shape[1], 1), dtype=torch.float32)
y_train = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
y_test = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

## 6. 训练模型与代码介绍

### 6.1 代码结构

训练部分代码分为以下模块：

| 模块 | 作用 |
|------|------|
| `TensorDataset` | PyTorch 数据集，将特征与标签 Tensor 封装为可索引的数据集 |
| `DataLoader` | 数据加载器，提供批量迭代、打乱顺序等功能 |
| `nn.MSELoss` | 均方误差损失函数，适用于回归问题 |
| `torch.optim.Adam` | Adam 优化器，自适应学习率 |
| 训练循环 | 手写前向传播 → 损失计算 → 反向传播 → 参数更新，流程清晰且易于控制 |

### 6.2 配置损失函数与优化器

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

epochs = 10
batch_size = 64

# 构建 PyTorch 数据集与数据加载器
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# 损失函数：均方误差（MSE），衡量预测值与真实值的差距
loss_fn = nn.MSELoss()

# 优化器：Adam，自适应调整学习率
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

print("训练配置:")
print(f"  Epochs: {epochs}")
print(f"  Batch Size: {batch_size}")
print(f"  Learning Rate: 0.001")
print(f"  Loss Function: MSELoss")
print(f"  Optimizer: Adam")

**超参数解读：**

| 参数 | 值 | 含义 |
|------|-----|------|
| `epochs` | 10 | 训练轮数，整个训练集遍历 10 次 |
| `batch_size` | 64 | 每次迭代喂入 64 个样本 |
| `learning_rate` | 0.001 | Adam 优化器的学习率 |
| `loss_fn` | MSELoss | 均方误差，衡量预测值与真实值的差距 |

### 6.3 启动训练

PyTorch 通过手写训练循环完成前向传播、损失计算、反向传播和参数更新。每个 epoch 内部遍历整个 DataLoader，逐 batch 计算梯度并更新参数。

In [ ]:
print("开始训练...")
net.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        pred = net(batch_x)           # 前向传播
        loss = loss_fn(pred, batch_y) # 计算损失

        optimizer.zero_grad()         # 清空梯度
        loss.backward()               # 反向传播
        optimizer.step()              # 更新参数

        epoch_loss += loss.item() * batch_x.size(0)
    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {epoch_loss / len(train_dataset):.6f}")
print("训练完成！")

训练过程中你会看到类似如下的输出：

```
Epoch [1/10], Loss: 0.051329
Epoch [2/10], Loss: 0.001017
Epoch [3/10], Loss: 0.000250
...
Epoch [10/10], Loss: 0.000273
```

- **loss（损失值）**：均方误差，衡量预测值与真实值的差距。随着训练进行，loss 应逐渐下降
- **epoch（轮次）**：当前训练进度

### 6.4 测试集预测与评估

训练完成后，在测试集上进行预测。预测结果需要**反归一化**——使用 `MinMaxScaler` 的 `inverse_transform` 将 `[0, 1]` 区间的预测值还原为原始价格，然后计算 RMSE（均方根误差）并绘制真实值与预测值的对比图。

RMSE 是回归任务常用的评估指标，计算公式为 `sqrt(mean((y_true - y_pred)^2))`，值越小说明模型预测越准确。

In [ ]:
# 进行预测
net.eval()
with torch.no_grad():
    predicted_scaled = net(X_test.to(device))

# 反归一化，还原为原始价格
predicted_prices = scaler.inverse_transform(predicted_scaled.cpu().numpy())
real_prices = scaler.inverse_transform(y_test.numpy())

# 绘制真实值与预测值对比图
plt.figure(figsize=(15, 6))
plt.plot(real_prices, color='blue', label='Real Stock Price')
plt.plot(predicted_prices, color='red', linestyle='--', label='Predicted Stock Price')
plt.title('Stock Price Prediction')
plt.xlabel('Time (Days)')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.show()

# 计算均方根误差 (RMSE)
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(real_prices, predicted_prices))
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")